# Surrogate Factory — UCAirfoils
## Chapter 9. Model Validation
Objectives:
- **9.0** Validate train/test split quality.
- **9.1** Predict on test set.
- **9.2** Compute metrics (R², MAE, quantile90).
- **9.2b** KS distribution tests: train vs test residuals.
- **9.3** Validate against requirements from SF_1.
- **9.4** Generate scatter and ratio plots.
- **9.5** Validation report (HTML).

### 0. Workflow initialisation

In [ ]:
import sys
from pathlib import Path
repo_root = str(Path('..').resolve().parent)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from IPython.display import display, HTML, JSON
from surrogate_factory.workflow import Workflow

workflow = Workflow('pipeline_config.yaml')
workflow.resume()

### 9. Model Validation

In [ ]:
workflow.import_metadata(stage_name='SF_9_Model_Validation')

In [ ]:
job = workflow.config['job_name']
Train_set = workflow.load_data(job + '_Train_set.csv')
Val_set   = workflow.load_data(job + '_Val_set.csv')
Test_set  = workflow.load_data(job + '_Test_set.csv')
print(f'Train set : {Train_set.shape}')
print(f'Val set   : {Val_set.shape}')
print(f'Test set  : {Test_set.shape}')

#### 9.0 Split Validation

In [ ]:
from model_validation.split_val import split_validation
split_result = split_validation(workflow, Train_set, Test_set)

#### 9.1 Predictions

In [ ]:
from model_validation.prediction import predict
model_output = predict(workflow, Test_set)
train_output = predict(workflow, Train_set)
val_output   = predict(workflow, Val_set)
model_output.head()

#### 9.2 Metrics

In [ ]:
from model_validation.score import calculate_metrics
metrics = calculate_metrics(workflow, Test_set, model_output)
JSON(metrics)

#### 9.2b Distribution Tests (KS: train vs test residuals)

In [ ]:
from model_validation.score import distribution_tests
ks_results = distribution_tests(workflow, Train_set, Test_set, train_output, model_output)

#### 9.3 Validation against requirements

In [ ]:
from model_validation.validation import validate
validate(workflow, metrics)

#### 9.3b Validation Summary — Table

In [ ]:
import ipywidgets as widgets
import pandas as pd
from IPython.display import display

validation_results = workflow.metadata.get_step_data(['metadata', 'Model_Validation', 'validation_results'])
models_info_w = workflow.metadata.get_step_data(['metadata', 'Model_Training', 'Models'])
labels_w = [m['label'] for m in models_info_w]

rows = []
for vr in validation_results:
    row = {'Output': vr['output'], 'Metric': vr['metric'], 'Target': f"< {vr['target']}"}
    for lbl in labels_w:
        m = vr['models'].get(lbl, {})
        score, passed = m.get('score'), m.get('passed')
        row[lbl] = (f"{'✅' if passed else '❌'} {score:.4f}") if score is not None else '?'
    rows.append(row)

df_val = pd.DataFrame(rows).set_index('Output')

def color_cell(val):
    if '✅' in str(val): return 'background-color:#d4edda; color:#155724'
    if '❌' in str(val): return 'background-color:#f8d7da; color:#721c24'
    return ''

styled = df_val.style.applymap(color_cell, subset=labels_w).set_caption('Validation Results')
display(styled)

#### 9.4 Plots

In [ ]:
%matplotlib inline
from model_validation.visualize import plot
plot(workflow, Test_set, model_output)

#### 9.5 Validation Report

In [ ]:
import os, sys, subprocess
from pathlib import Path
from model_validation.export_validation_csvs import export_validation_csvs

csv_dirs = export_validation_csvs(
    workflow,
    Train_set, Val_set, Test_set,
    train_output, val_output, model_output,
)

script_path = Path(workflow.config["data.folder"]).parent / "python_nodes_library" / "model_validation" / "validation_script.py"
output_dir  = Path(workflow.config["artifacts.folder"]) / "validation_reports"
output_dir.mkdir(parents=True, exist_ok=True)

ms_w = workflow.metadata.get_step_data(["metadata", "Model_Selection"])
num_inputs_w = Train_set[ms_w["inputs"]].select_dtypes(include="number").columns.tolist()

# Repo root = .../<repo>/<UseCase>/pipeline/python_nodes_library/model_validation -> up 4.
# Derived, never hardcoded, so the notebook runs on any machine.
repo_root = script_path.parents[4]
env = {
    **os.environ,
    "PYTHONPATH": str(repo_root / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
    "MLFLOW_ALLOW_FILE_STORE": "true",
}

# The report template runs voxel tesselation + convex-hull membership, which is
# O(n^2) in the number of unique input combinations. Above a few thousand rows
# it never finishes, so cap the sample the template works on.
subsample = 300 if len(Train_set) > 5000 else None
print(f"Train rows: {len(Train_set):,} -> subsample_size = {subsample or 'full set'}")

for label, csv_dir in csv_dirs.items():
    print(f"Running validation report for {label}...")
    cmd = [
        sys.executable, str(script_path),
        "-d", str(csv_dir), "-n", label, "-o", str(output_dir),
        "--exclude_warnings",
    ]
    if subsample:
        cmd += ["--subsample_size", str(subsample)]
    cmd += ["--splitting_variables", *num_inputs_w]

    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800, env=env)
        if result.returncode == 0:
            print(f"  OK  {output_dir / (label + '_validation_output.html')}")
        else:
            print(f"  FAILED\n{result.stderr[-2000:]}")
    except subprocess.TimeoutExpired:
        print(f"  TIMEOUT (>1800s) for {label} - lower subsample further.")


### Save

In [ ]:
workflow.save_metadata()

#### 9.7 Executive Summary Report (PDF + HTML)

Generates the mandatory multi-part validation report via
`reporting/generate_executive_summary.py`:

- **Part 1 - Executive Summary** (winner model, <= 2 pages)
- **Part 2 - Technical Review** (all models: TOC, scatter/ratio, KS, split, roadmap)
- **Part 3 - Deep Analysis** (winner model: data overview, train-test split,
  error quantification, P(E|X), P(E|Y), uncertainty)

Output lands in `data/artifacts/validation_reports/`.

> **Needs a LaTeX engine.** `tectonic` is preferred but is **not available via pip**.
> Install the static binary (no root, no conda):
> ```
> mkdir -p ~/.local/bin && cd ~/.local/bin
> curl --proto "=https" --tlsv1.2 -fsSL https://drop-sh.fullyjustified.net | sh
> export PATH="$HOME/.local/bin:$PATH"
> ```
> Any TeX Live install also works - the report falls back to
> `latexmk`, `xelatex` or `pdflatex` automatically.


In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

pipeline_dir  = Path(workflow.config["data.folder"]).parent
artifacts_dir = Path(workflow.config["artifacts.folder"])
job           = workflow.config["job_name"]

report_script = pipeline_dir / "python_nodes_library" / "reporting" / "generate_executive_summary.py"
meta_json     = artifacts_dir / f"metadata_{job}.json"
report_dir    = artifacts_dir / "validation_reports"
report_dir.mkdir(parents=True, exist_ok=True)

# Repo root = .../<repo>/<UseCase>/pipeline -> up 2. Keeps this portable across machines.
repo_root = pipeline_dir.parent.parent
env = {
    **os.environ,
    "PYTHONPATH": str(repo_root / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
    "MLFLOW_ALLOW_FILE_STORE": "true",
}

engines = [e for e in ("tectonic", "latexmk", "xelatex", "pdflatex") if shutil.which(e)]
print(f"LaTeX engines available: {', '.join(engines) if engines else 'NONE'}")

if not report_script.exists():
    print(f"Report script not found: {report_script}")
elif not meta_json.exists():
    print(f"Metadata not found: {meta_json}\n   Run the 'Save' cell above first.")
else:
    try:
        result = subprocess.run(
            [sys.executable, str(report_script), str(meta_json), "--output", str(report_dir)],
            capture_output=True, text=True, timeout=1800, env=env,
        )
        if result.stdout:
            print(result.stdout[-4000:])
        if result.returncode == 0:
            for f in sorted(report_dir.glob("executive_summary_*")) + \
                     sorted(report_dir.glob("deep_analysis_*")):
                print(f"  OK  {f.name}  ({f.stat().st_size/1024:.0f} KB)")
        else:
            print("Report generation failed:")
            print(result.stderr[-3000:])
    except subprocess.TimeoutExpired:
        print("Timeout (>1800s) building the executive summary report.")
